# Semantic Chunking Experiment

This notebook explores semantic chunking, a technique that splits text based on meaning rather than fixed token limits or simple punctuation.

The pipeline is:
Passage -> Sentence segmentation -> Sentence embeddings -> Adjacent similarity -> Boundary detection -> Chunks

We'll compare semantic chunking with our baseline strategies to see if the extra computational cost is justified.

In [ ]:
import sys
import os

# Import from colab/src/
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..", "src")))
from utils import load_config, set_seed, find_repo_root, save_json, get_reports_dir
from dataset_utils import load_msmarco_xi, get_selected_passages
from chunking import chunk_semantic, split_sentences, chunk_metadata_aware, compare_strategies, run_chunking
from embeddings import EmbeddingModel
from retrieval import FAISSIndex
from evaluation import evaluate_retrieval, compare_configurations

repo_root = find_repo_root()
config = load_config()
set_seed(42)


## 1. Setup Encoding Function and Data
Semantic chunking requires sentence embeddings to measure similarity between adjacent sentences.

In [ ]:
df = load_msmarco_xi(repo_root)
passages = get_selected_passages(df)
sample_passages = passages[:100]

# Load embedding model for sentence encoding
model_name = config.get('embeddings', {}).get('model_name', 'all-MiniLM-L6-v2')
emb_model = EmbeddingModel(model_name)

def encode_fn(sentences):
    return emb_model.encode(sentences)


## 2. Experiment with Similarity Thresholds
We will try different thresholds (0.3, 0.5, 0.7) to see how they affect chunk boundaries.

In [ ]:
thresholds = [0.3, 0.5, 0.7]
semantic_results = {}
test_text = sample_passages[0]['text']

for t in thresholds:
    chunks = chunk_semantic(test_text, encode_fn, similarity_threshold=t)
    semantic_results[t] = chunks
    print(f"Threshold {t}: {len(chunks)} chunks generated")
    if chunks:
        print(f"  First chunk size: {len(chunks[0].text)} chars")


## 3. Compare with Baseline Strategies
Let's compare semantic chunks against other chunking strategies (like sentence-aware) on our sample set.

In [ ]:
all_chunks = []
for p in sample_passages:
    p_chunks = chunk_semantic(p['text'], encode_fn, similarity_threshold=0.5)
    for c in p_chunks:
        c.metadata['doc_id'] = p['id']
    all_chunks.extend(p_chunks)

print(f"Total semantic chunks generated: {len(all_chunks)}")


## 4. Evaluate Retrieval Quality
We build a FAISS index from our semantic chunks to evaluate retrieval quality.

In [ ]:
index = FAISSIndex(emb_model.dimension)
chunk_texts = [c.text for c in all_chunks]
chunk_embeddings = emb_model.encode(chunk_texts)
index.build(chunk_embeddings, all_chunks)

# Using dummy evaluation queries for this demonstration.
queries = [{"id": 1, "text": "dummy query"}]
qrels = {"1": {sample_passages[0]['id']: 1}}

retrieval_eval = evaluate_retrieval(index, emb_model, queries, qrels, k_values=[5, 10])
print(f"Retrieval Results: {retrieval_eval.metrics}")


## 5. Comparison and Saving Results
We'll save the evaluation metrics to compare retrieval results between semantic and sentence-aware chunking.

In [ ]:
reports_dir = get_reports_dir(repo_root)
output_path = os.path.join(reports_dir, 'semantic_chunking_results.json')
save_json(retrieval_eval.to_dict(), output_path)
print(f"Results saved to {output_path}")


## Decision: Should we use Semantic Chunking?

Based on the evidence presented in our evaluation metrics, we need to balance the computational cost of running embedding models for boundary detection against the gains in retrieval quality (Recall@K / MRR).

If semantic chunking does not significantly outperform sentence-aware chunking, we should fall back to metadata-aware sentence chunking for production due to its lower latency.